In [ ]:
from unsloth import FastLanguageModel

def load_model(model_name: str, max_sequence_length: int = 2048, load_in_4bit: bool = True):

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_sequence_length,
        load_in_4bit = load_in_4bit,  #
        # token = "YOUR_HF_TOKEN", # HF Token for gated models
    )
    return model, tokenizer


model, tokenizer = load_model("unsloth/mistral-7b-instruct-v0.3-bnb-4bit")


==((====))==  Unsloth 2026.2.1: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    Tesla V100-PCIE-16GB. Num GPUs = 1. Max memory: 15.773 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
<s>[INST] Schrijf een kort gedicht dat de schoonheid van 'X' beschrijft.v
input: X: Een meer[/INST] Een meer, die in de zon glanst,
Een spiegel voor de hemel,
Een schat van natuurlijke kansen,
Waar de tijd stil verstelt.

Een spiegel voor de sterren,
Een schoonheid in de rust,
Een verlangen voor de zee,
Een liefde voor het best.

Een meer, die in de nacht,
Een spookachtige glans geeft,
Een verhaal van verleden,
Een toekomst in de dromen.

Een meer, die in de zomer,
Een koele ontspanning biedt,
Een plek van vreugde en rust,
Een sc

In [4]:
user_question = "Adviseer iemand hoe te ontstressen"

messages = [
    {"role": "user", "content": user_question},
]
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")



outputs = model.generate(
    inputs,
    max_new_tokens=600,                  # hard stop
    #eos_token_id=tokenizer.eos_token_id, # stop on EOS
    #pad_token_id=tokenizer.pad_token_id,
    #use_cache=True,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=False))

<s>[INST] Adviseer iemand hoe te ontstressen[/INST] Om iemand te adviseren hoe te ontstressen, kan ik de volgende tips geven:

1. Breken van de dag: Maak een ritme in je dag waarbij je regelmatig pauzes neemt om te rusten en te ontspannen.
2. Lichamelijke activiteit: Doe regelmatig lichamelijke activiteit, zoals wandelen, fietsen of yoga.
3. Goede slaap: Zorg ervoor dat je goed slaapt en volgt een goede slaaphygiëne.
4. Eet gezond: Eet gezond en voedzuchtig en voorkom zo veel mogelijk stress-eiwitten in je dieet.
5. Relaxatie: Probeer verschillende vormen van relaxatie uit, zoals meditatie, autogenes training of progressieve relaxatie.
6. Sociale contacten: Versterk je sociale contacten en praat met mensen die je liefhebben.
7. Leren om te zeggen nee: Leer om je grenzen te zetten en om te zeggen nee aan verplichtingen die je niet wilt of kan volbrengen.
8. Verwachten en accepteren: Verwacht dat er stress in je leven zal komen en accepteer dat je niet altijd controle hebt over je omgevi

In [13]:

user_question = "Explain AI in 2 phrases"

messages = [
    {"role": "user", "content": user_question},
]
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")



outputs = model.generate(
    inputs,
    max_new_tokens=600,                  # hard stop
    eos_token_id=tokenizer.eos_token_id, # stop on EOS
    #pad_token_id=tokenizer.pad_token_id,
    #use_cache=True,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=False))

<s>[INST] Explain AI in 2 phrases[/INST] 1. AI is a branch of computer science that enables machines to mimic intelligent human behavior.

2. AI uses algorithms and statistical models to analyze data, make decisions, and solve complex problems.</s>


In [ ]:
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default"
)

client = OpenAI(  
  base_url = "https://finetuning-foundry.openai.azure.com/openai/v1/",  
  api_key=token_provider,
)

response = client.chat.completions.create(
  model="grok-4-fast-reasoning",  # Replace with your model deployment name.
    messages=[
        {"role": "system", "content": "Assistant is a large language model trained by OpenAI."},
        {"role": "user", "content": "Who were the founders of Microsoft?"}
    ]
)

#print(response)
print(response.model_dump_json(indent=2))
print(response.choices[0].message.content)

In [5]:
from datasets import load_from_disk

alpaca_train_formatted = load_from_disk("datasets/alpaca_train_formatted", keep_in_memory=True)

In [7]:
alpaca_test_formatted = load_from_disk("datasets/alpaca_test_formatted", keep_in_memory=True)

In [8]:
alpaca_train_formatted[0]

{'text': 'Beantwoord de volgende vraag zo goed mogelijk.\n\n### Vraag:\nSchrijf een gedicht van 7 regels met behulp van de gegeven woorden\ninput: paraplu, rail, water, lucht, gras, blad\n\n### Antwoord:\nOnder de donker wordende lucht,\nHet gras groeit nog steeds.\nEen eenzaam blad drijft weg\nVan de grijze ijzeren rail,\nDobberend als een paraplu in de golven van water.\nLangzaam dringt de nacht binnen, brengt rust en kalmte.\nEn ik sta en adem, voel de vrede in deze balsem.\n</s>'}